# RSNA Knee — label diagnostics (CPU, ~1 min)
キーワード規則のラベルを、正解ラベル付き58件と突き合わせる。本文は出さず、語の文書頻度だけを出す。規則の改善点を探すためのもの。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
SRC = Path('/kaggle/working/repo/rsna-knee-abnormality-detection/src')
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb','/kaggle/working/repo'], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
train = pd.read_csv(COMP/'train.csv')
train['has_true'] = train['ACL'].notna()
train = add_pseudo_labels_ml(train)
tv = train[train.has_true].reset_index(drop=True)
print('true-labeled:', len(tv), tv.lang.value_counts().to_dict())
from multilingual_labels import label_report_lang
R = np.array([[label_report_lang(r, l)[c] for c in LABELS] for r, l in zip(tv.Report.fillna(''), tv.lang)])   # 正解を見ずに規則だけで作ったラベル
T = tv[LABELS].values.astype(int)
tp, fp, fn = (R & T).sum(0), (R & (1 - T)).sum(0), ((1 - R) & T).sum(0)
rest = train[~train.has_true & train.lang.isin(['en', 'tr', 'el', 'bg', 'es', 'de', 'fr'])]
print(pd.DataFrame({'true_prev(58)': T.mean(0).round(2), 'rule_prev(rest)': rest[LABELS].mean().values.round(2),
                    'TP': tp, 'FP': fp, 'FN': fn, 'precision': (tp / np.maximum(tp + fp, 1)).round(2),
                    'recall': (tp / np.maximum(tp + fn, 1)).round(2)}, index=LABELS))
for grp, m in (('en', tv.lang == 'en'), ('non-en', tv.lang != 'en')):
    m = m.values
    print(grp, 'n=%d' % m.sum(), 'micro precision %.3f recall %.3f' % (
        (R[m] & T[m]).sum() / max(R[m].sum(), 1), (R[m] & T[m]).sum() / max(T[m].sum(), 1)))

In [ ]:
import re
from collections import Counter
docs = [set(re.findall(r'[^\W\d_]{4,}', str(t).lower())) for t in tv.Report.fillna('')]
def df(idx):
    c = Counter()
    for i in idx: c.update(docs[i])
    return c
def contrast(a, b, n, min_df):
    da, db = df(a), df(b)
    sc = {w: da[w] / max(len(a), 1) - db[w] / max(len(b), 1) for w in da if da[w] >= min_df}
    return [(w, da[w], db[w]) for w in sorted(sc, key=sc.get, reverse=True)[:n]]
for j, c in enumerate(LABELS):
    pos, neg = np.where(T[:, j] == 1)[0], np.where(T[:, j] == 0)[0]
    if len(pos) < 5 or len(neg) < 5: continue
    print(f'### {c}  (true+ {len(pos)}, true- {len(neg)})')
    print(' words in TRUE positives vs negatives (word, df_pos, df_neg):', contrast(pos, neg, 15, 3))
    fn_i, tp_i, fp_i = (np.where((T[:, j] == 1) & (R[:, j] == 0))[0], np.where((T[:, j] == 1) & (R[:, j] == 1))[0],
                        np.where((T[:, j] == 0) & (R[:, j] == 1))[0])
    if len(fn_i) >= 3: print(f' missed by rules (FN={len(fn_i)}) vs found (word, df_fn, df_tp):', contrast(fn_i, tp_i, 10, 2))
    if len(fp_i) >= 3: print(f' false alarms (FP={len(fp_i)}) vs correct (word, df_fp, df_tp):', contrast(fp_i, tp_i, 10, 2))